# Solar Filament Segmentation Challenge 2026 — EDA + RF-DETR-Seg ⚡PTL

This notebook studies the MAGFiLO H-Alpha observations and trains a reproducible RF-DETR instance-segmentation
baseline.

- **Why RF-DETR-Seg**: the challenge scores one mask per filament, requires mask RLE counts, and penalizes
  fragmented or merged instances.
- **Pipeline**: converts the provided COCO-style polygons into RF-DETR's public PyTorch Lightning
  data/model/trainer building blocks, snaps each predicted mask onto the dark filament pixels it covers
  (intensity-guided refinement — filaments are dark absorption features), then validates a submission-shaped CSV.

## 1. Environment setup

- **Offline-first**: prefer the `frozen_packages` cache so a submission run stays reproducible without live
  internet; fall back to an online install when the cache is absent.
- **Version pin**: `rfdetr>=1.8.3` — the version every API in this notebook was verified against (`RFDETRSegSmall`,
  `convert_coco_poly_to_mask`, `RFDETRDataModule._show_samples`, `rfdetr.visualize.plot_metrics` all confirmed
  present at 1.8.3).
- **Extras**: `train` covers `pytorch_lightning`/`torchmetrics`/`pycocotools`/`scipy`; `visual` adds
  `matplotlib`/`pandas`/`seaborn`, required by `_show_samples()` and `plot_metrics()`'s seaborn error bands.
  `supervision`/`torchvision` are already core `rfdetr` dependencies, not extras.
- **`--upgrade` on `transformers` is required, not cosmetic**: Kaggle base images ship a `transformers` build that
  version-parses as satisfying RF-DETR's `transformers>=5.1.0,<6.0.0` pin but is missing
  `BackboneConfigMixin`/`BackboneMixin` at the top-level package (`ImportError` from
  `rfdetr.models.backbone.dinov2_with_windowed_attn`). Without `--upgrade`, pip treats that preinstalled copy as
  already satisfying the pin and never replaces it.
- **Kernel restart**: if `transformers` was already imported earlier in this kernel (e.g. by a Kaggle bootstrap
  cell), restart the kernel after this cell so the upgraded copy is actually loaded.

In [ ]:
! pip download -q "rfdetr[train,loggers,visual]>=1.8.3" transformers --dest frozen_packages
! pip install -q --no-index --find-links frozen_packages --upgrade "rfdetr[train,loggers,visual]>=1.8.3" transformers  # noqa: E501
! pip install -q --upgrade "rfdetr[train,loggers,visual]>=1.8.3" transformers  # online fallback

## 2. Imports, paths, and reproducibility

Resolves dataset/output paths (env-overridable for local vs Kaggle runs), seeds every RNG, and fails fast if the
expected MAGFiLO directory layout is missing.

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import json
import os
import warnings
from collections.abc import Callable
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from pycocotools import mask as coco_mask
from pytorch_lightning import seed_everything
from pytorch_lightning.callbacks import LearningRateMonitor
from rfdetr import RFDETRSegSmall
from rfdetr.config import RFDETRSegSmallConfig, SegmentationTrainConfig
from rfdetr.datasets.coco import convert_coco_poly_to_mask, make_coco_transforms_square_div_64
from rfdetr.training import RFDETRDataModule, RFDETRModelModule, build_trainer
from rfdetr.visualize import plot_metrics
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.segmentation import DiceScore
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning, module="albumentations.check_version")

# ==============================
# Grounded paths and reproducibility
SEED = 42
PATH_DATASET = Path(
    os.environ.get(
        "FILAMENT_DATASET_DIR",
        "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026",
    )
)
PATH_OUTPUT = Path(os.environ.get("FILAMENT_OUTPUT_DIR", "."))
PATH_MODELS = PATH_OUTPUT / "models"
PATH_RFDETR_CACHE = Path.home() / ".roboflow" / "models"
PATH_TRAIN_IMAGES = PATH_DATASET / "train" / "train_images"
PATH_TEST_IMAGES = PATH_DATASET / "test" / "test_images"
PATH_ANNOTATIONS = PATH_DATASET / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"


def resolve_device() -> torch.device:
    """Pick the best available torch device: CUDA (Kaggle/most GPUs), then Apple MPS, else CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = resolve_device()

# Smoke run: exercise the whole pipeline (train -> calibrate -> submit) in minutes to validate a Kaggle commit
# won't crash, before spending hours on a full run. Enable with FILAMENT_SMOKE_RUN=1. It caps the batches per
# epoch and shrinks the calibration/test image counts; it does NOT produce a competitive submission.
SMOKE_RUN = os.environ.get("FILAMENT_SMOKE_RUN", "0") == "1"
SMOKE_BATCHES = 3

seed_everything(SEED, workers=True)
PATH_OUTPUT.mkdir(parents=True, exist_ok=True)
PATH_RFDETR_CACHE.mkdir(parents=True, exist_ok=True)

for package_name in ("torch", "pytorch-lightning", "rfdetr", "pycocotools"):
    print(f"{package_name}=={metadata.version(package_name)}")
print(f"device={DEVICE}; smoke_run={SMOKE_RUN}")
print(f"PATH_DATASET={PATH_DATASET.resolve()}")
print(f"PATH_OUTPUT={PATH_OUTPUT.resolve()}")
print(f"PATH_RFDETR_CACHE={PATH_RFDETR_CACHE}")

if not PATH_ANNOTATIONS.is_file() or not PATH_TRAIN_IMAGES.is_dir() or not PATH_TEST_IMAGES.is_dir():
    raise FileNotFoundError(
        "Set FILAMENT_DATASET_DIR to the extracted MAGFiLO_1.0_Kaggle_2026 directory. "
        f"Expected {PATH_ANNOTATIONS}, {PATH_TRAIN_IMAGES}, and {PATH_TEST_IMAGES}."
    )

## 3. Sample images and annotation statistics

- Loads the organizer's COCO document into two dataframes.
- Runs cheap correctness guards: unique image ids, no orphan annotations, sampled files present.
- Shows a few raw images, then reports annotation statistics.
- **Grayscale note**: GONG H-Alpha JPEGs are single-channel grayscale (documented in `data.md`). Images stay
  untouched here and are only replicated to three channels later, right before they reach the ImageNet-pretrained
  RF-DETR backbone.

In [ ]:
SAMPLE_N = 9
EDA_FILE_CHECK_N = 64
TARGET_COL = "segmentation"
IMAGE_HEIGHT = 2048
IMAGE_WIDTH = 2048


def load_coco_document(annotation_path: Path) -> dict[str, Any]:
    """Load a COCO document and validate the fields this baseline consumes."""
    with annotation_path.open(encoding="utf-8") as file:
        document = json.load(file)
    required_keys = {"images", "annotations", "categories"}
    missing_keys = required_keys.difference(document)
    if missing_keys:
        raise KeyError(f"COCO document is missing required keys: {sorted(missing_keys)}")
    if not document["images"] or not document["annotations"]:
        raise ValueError("COCO training document has no image records or annotations.")
    return document


coco_document = load_coco_document(PATH_ANNOTATIONS)
df_images = pd.DataFrame(coco_document["images"]).rename(columns={"id": "source_image_id"})
df_annotations = pd.DataFrame(coco_document["annotations"])
category_names = {category["id"]: category["name"] for category in coco_document["categories"]}

df_images["source_image_id"] = df_images["source_image_id"].astype(str)
df_annotations["source_image_id"] = df_annotations["image_id"].astype(str)
df_annotations["category_name"] = df_annotations["category_id"].map(category_names).fillna("unknown")
df_annotations["area_fraction"] = df_annotations["area"] / (IMAGE_HEIGHT * IMAGE_WIDTH)
annotation_counts = df_annotations.groupby("source_image_id").size().rename("annotation_count")
df_images = df_images.merge(annotation_counts, how="left", left_on="source_image_id", right_index=True)
df_images["annotation_count"] = df_images["annotation_count"].fillna(0).astype(int)

if df_images["source_image_id"].duplicated().any():
    raise ValueError("COCO image ids must be unique before building the RF-DETR split.")
if not df_annotations["source_image_id"].isin(df_images["source_image_id"]).all():
    raise ValueError("At least one annotation references a missing image record.")
if df_annotations[TARGET_COL].isna().any():
    raise ValueError("At least one annotation has no segmentation polygon.")

checked_files = [
    (PATH_TRAIN_IMAGES / file_name).is_file()
    for file_name in tqdm(
        df_images["file_name"].head(EDA_FILE_CHECK_N),
        desc="check image files",
        leave=False,
    )
]
if not all(checked_files):
    raise FileNotFoundError(f"{checked_files.count(False)} sampled training images are missing.")
print(f"images={len(df_images):,}; annotations={len(df_annotations):,}; categories={len(category_names)}")
display(df_images.head())
display(df_annotations[["category_name", "area", "bbox", "iscrowd"]].head())

In [ ]:
def show_sample_images(images: pd.DataFrame, n: int) -> None:
    """Display n raw grayscale training images, no annotation overlay."""
    sampled = images.sample(n=min(n, len(images)), random_state=SEED)
    rows = int(np.ceil(len(sampled) / 3))
    _, axes = plt.subplots(rows, 3, figsize=(15, 5 * rows))
    axes_flat = np.atleast_1d(axes).ravel()
    for axis, image_row in zip(axes_flat, sampled.itertuples(index=False), strict=False):
        with Image.open(PATH_TRAIN_IMAGES / image_row.file_name) as image:
            axis.imshow(image, cmap="gray")
        axis.set_title(image_row.file_name, fontsize=8)
        axis.axis("off")
    for axis in axes_flat[len(sampled) :]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()


show_sample_images(df_images, SAMPLE_N)

### Annotation statistics

Category counts, area distribution, grayscale/2048x2048 sanity checks, and the repeated-physical-JPEG count that
motivates the grouped train/validation split in section 4.

In [ ]:
category_counts = df_annotations["category_name"].value_counts().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
_ = category_counts.plot(kind="bar", ax=axes[0], color="tab:blue")
axes[0].set_title("Annotation categories")
axes[0].set_xlabel("Source category")
axes[0].set_ylabel("Filament instances")
axes[0].grid(True, axis="y")

_ = axes[1].hist(np.log10(df_annotations["area"].clip(lower=1)), bins=40, color="tab:orange")
axes[1].set_title("Filament annotation area")
axes[1].set_xlabel("log10(mask area in pixels)")
axes[1].set_ylabel("Filament instances")
axes[1].grid(True)
plt.tight_layout()
plt.show()

image_size_records: list[dict[str, Any]] = []
for file_name in tqdm(
    df_images["file_name"].head(EDA_FILE_CHECK_N),
    desc="read image metadata",
    leave=False,
):
    with Image.open(PATH_TRAIN_IMAGES / file_name) as image:
        image_size_records.append({"file_name": file_name, "mode": image.mode, "size": image.size})
df_image_sizes = pd.DataFrame(image_size_records)
if set(df_image_sizes["size"]) != {(IMAGE_WIDTH, IMAGE_HEIGHT)}:
    raise ValueError("Sampled image dimensions differ from the documented 2048x2048 submission shape.")

empty_annotation_records = int((df_images["annotation_count"] == 0).sum())
repeated_files = int(df_images["file_name"].duplicated(keep=False).sum())
repeated_file_groups = int((df_images["file_name"].value_counts() > 1).sum())
print(f"stats — sampled image mode(s): {sorted(df_image_sizes['mode'].unique())}; size={IMAGE_WIDTH}x{IMAGE_HEIGHT}")
print(f"stats — empty-annotation image records: {empty_annotation_records:,}/{len(df_images):,}")
print(
    "stats — area_fraction median/p95: "
    f"{df_annotations['area_fraction'].median():.5f}/{df_annotations['area_fraction'].quantile(0.95):.5f}"
)
print(f"stats — records sharing a physical JPEG: {repeated_files:,} across {repeated_file_groups:,} file names")
print("note — GroupShuffleSplit groups by file_name below, so one physical observation never crosses splits.")

## 4. Train/validation split

- **Grouping key**: `GroupShuffleSplit` assigns physical JPEGs — not COCO image records — to train or validation,
  so an observation annotated more than once never crosses the split.
- **Train-side annotator dedup**: after the grouped split, the train split keeps exactly one record (the
  most-annotated one) per physical JPEG so the model never sees the same pixels with conflicting ground truth;
  validation keeps all annotator copies. Rationale inline in `FilamentDataModule.__init__`.
- **No COCO JSON roundtrip**: Section 6 reads directly from `df_images`/`df_annotations`; no intermediate COCO
  JSON view is written to or re-parsed from disk.

In [ ]:
VALIDATION_FRACTION = 0.15

### Why 768px and how the resolution ladder works

Square input preserves thin filament barbs; RF-DETR requires a multiple of `patch_size*num_windows`. Seg-Nano
uses `patch_size=12`, `num_windows=1` → divisor 12 (verified against
`rfdetr.models.backbone.dinov2_with_windowed_attn`, which raises `ValueError` on a non-multiple). Valid
up/down-sizing ladder (all multiples of 12): 360, 480, 576, 672, 768, 900, 1032 — move one step down on OOM,
one step up if there's headroom.
#
**Why resolution is the score bottleneck here**: the previous 576px run reached only ~0.41 union Dice while
plain semantic-segmentation baselines reach ~0.7. The median filament covers ~0.03% of the 2048x2048 frame —
thin, elongated structures a few pixels wide. Every resolution step the pipeline drops (2048 → input resolution
→ mask-head grid, see the mask-grid note in section 5) erases barbs first, and the Dice score is won exactly
on those thin boundaries. 900px OOM'd on a 16 GB T4 at `batch_size=2`; 768px at `batch_size=1` with gradient
accumulation keeps the same effective batch while spending the memory budget on resolution instead.

In [ ]:
TRAINING_RESOLUTION = 768
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
NUM_WORKERS = 2

## 5. RF-DETR-Seg model and public PTL primitives

- **Model + resolution**: RF-DETR-Seg-Nano at `TRAINING_RESOLUTION` (768px, see section 4) — the 2048-pixel solar
  images carry thin barbs and fine filament threads that a small 384-pixel input erases, so the extra resolution
  is spent where the Dice score is won. Nano is the lightest variant, chosen to fit a 16 GB T4 at this resolution.
- **Mask-head grid**: RF-DETR's segmentation head predicts mask logits at `resolution / mask_downsample_ratio`
  and bilinearly upsamples them to the image size at postprocess time (verified in
  `rfdetr/models/heads/segmentation.py` and `rfdetr/models/postprocess.py`). At the default ratio 4 a 576px run
  predicts masks on a 144px grid — the median filament shrinks to roughly 8x1 grid cells there, and barbs
  vanish entirely, which caps the achievable Dice near 0.5 regardless of training length. This notebook sets
  `mask_downsample_ratio=2`, doubling the mask grid (768px input → 384px mask grid) for a seg-head-only memory
  cost. Expect a `PretrainWeightsCompatibilityWarning` from this override — it is over-conservative here: the
  ratio only changes the mask head's interpolation target size (`rfdetr/models/heads/segmentation.py`), no
  parameter shape depends on it, so every published weight still loads and fine-tuning adapts the convolutions
  to the finer grid.
- **Trading speed for accuracy**: swap `RFDETRSegSmall`/`RFDETRSegSmallConfig`/`rf-detr-seg-small.pt` (or the
  Medium/Large variants) on a bigger GPU.
- **Expected load warning, verified harmless**: loading `rf-detr-seg-nano.pt` logs
  "checkpoint lacks args.num_queries / args.group_detr; falling back to flat slice ... may scramble per-group
  query structure". Scrambling only occurs when the configured `num_queries * group_detr` is *smaller* than the
  checkpoint's query tensor (verified in `rfdetr/models/weights.py:_slice_query_param_per_group`). Here both are
  100 x 13 = 1300 rows, so the flat slice copies the tensor unchanged — pretrained query structure is intact.
- **Training stack**: the public `RFDETRModelModule` owns AdamW, scheduler, mask Dice loss, and mixed precision.
- **Data stack**: the custom DataModule inherits RF-DETR's public loader and collate implementation, then binds
  it directly to the source annotation dataframes — no second COCO JSON, no copied images, Lightning callbacks
  and checkpoints stay visible throughout.

### Why EMA is enabled (and the checkpoint-provenance caveat that once disabled it)

`RFDETREMACallback` keeps an exponential moving average of the weights, which usually generalizes slightly
better than the live weights — a free accuracy gain over a 40-epoch run, so this notebook enables it. The
caveat that previously kept it off: EMA weights can silently win `checkpoint_best_total.pth` over the live
model (`BestModelCallback.on_fit_end`, strict `>` favors EMA) with no field in the saved checkpoint recording
which one it is (verified in rf-detr source). Consequence — the reloaded best checkpoint below may legitimately
differ from the in-memory `module.model`, and the smoke test treats that as expected behavior, not a bug. All
scored predictions use the reloaded checkpoint only, so provenance stays unambiguous where it matters.

### Epoch budget — bounded by Kaggle wall clock

The first 10-epoch run stopped after ~180 optimizer steps (effective batch 64 on ~1k image records) with
validation segmentation mAP still rising at the cut — classic undertraining. Effective batch is now 16
(`1 x 16`), which gives far more optimizer steps per epoch. The hard constraint is wall clock: at 768px an
epoch measured ~45 min on a T4, so a Kaggle GPU session (~12 h) affords roughly 12 epochs once inference and
calibration are accounted for. `MAX_EPOCHS` is set accordingly and early stopping (patience 10) trims it if
validation mAP plateaus sooner. To push further, either lower `TRAINING_RESOLUTION` one ladder step (section 4)
to buy faster epochs, or split training across sessions by resuming from `checkpoint_best_total.pth`.
#
**Smoke run**: when `SMOKE_RUN` is set (section 2), `MAX_EPOCHS` drops to 2 and each epoch is capped to
`SMOKE_BATCHES` train/val batches (wired into the trainer in section 6), so the whole notebook finishes in
minutes to prove it runs end to end without crashing.

In [ ]:
MODEL_NAME = "RFDETRSegSmall"
MAX_EPOCHS = 2 if SMOKE_RUN else 6
PRETRAINED_WEIGHTS = "rf-detr-seg-small.pt"
LEARNING_RATE = 1e-4
EARLY_STOPPING_PATIENCE = 10
MASK_DOWNSAMPLE_RATIO = 2  # mask grid = resolution / ratio; see the mask-head grid note above
FAST_DEV_RUN = False

model_config = RFDETRSegSmallConfig(
    num_classes=1,
    resolution=TRAINING_RESOLUTION,
    pretrain_weights=PRETRAINED_WEIGHTS,
    mask_downsample_ratio=MASK_DOWNSAMPLE_RATIO,
)
train_config = SegmentationTrainConfig(
    dataset_file="coco",
    dataset_dir=str(PATH_TRAIN_IMAGES),
    output_dir=str(PATH_MODELS),
    class_names=["filament"],
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LEARNING_RATE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    early_stopping=True,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    eval_interval=1,
    use_ema=True,  # see "Why EMA is enabled" above
    run_test=False,
    tensorboard=True,
    progress_bar="tqdm",
    augmentation_backend="cpu",
    mask_dice_loss_coef=5.0,
)
print(f"model={MODEL_NAME}; resolution={model_config.resolution}; effective_batch={BATCH_SIZE * GRAD_ACCUM_STEPS}")
print("Primary selection limit — RF-DETR's built-in best checkpoint tracks validation segmentation mAP, not Dice.")
print("Recovery — the next section calibrates confidence with a matched-instance Dice proxy on grouped validation.")

## 6. Lightning training and data lens

- **Trainer stack**: RF-DETR's public trainer builds the Lightning `CSVLogger`, COCO evaluation, best-model
  checkpointing, and optional early stopping stack (EMA disabled, see section 5). A learning-rate monitor is
  appended after construction, preserving those RF-DETR callbacks.
- **Data lens**: inspects an actual transformed batch before fitting so shape, dtype, and instance-mask errors
  surface before a longer GPU run.
- **Single-GPU**: this interactive notebook explicitly uses one GPU; Lightning selects its CUDA single-device
  strategy from that device count. Launch a script with `torchrun` for reliable multi-GPU DDP.

In [ ]:
class FilamentDataset(torch.utils.data.Dataset):
    """Build one RF-DETR sample per row of a source-dataframe image split (no COCO JSON roundtrip)."""

    def __init__(
        self,
        images: pd.DataFrame,
        annotations: pd.DataFrame,
        image_directory: Path,
        transforms: Callable[[Image.Image, dict[str, Any]], tuple[Any, dict[str, Any]]] | None,
    ) -> None:
        self._images = images.reset_index(drop=True)
        self._annotations_by_image = dict(tuple(annotations.groupby("source_image_id")))
        self._image_directory = image_directory
        self._transforms = transforms
        # Best-effort hooks for RFDETRDataModule._show_samples()'s filename lookup, which also requires a `.coco`
        # attribute this dataset intentionally has none of; it falls back to a positional title without one.
        self.root = image_directory
        self.ids = self._images["source_image_id"].tolist()

    def __len__(self) -> int:
        """Return the number of image records in this split."""
        return len(self._images)

    def __getitem__(self, index: int) -> tuple[Any, dict[str, Any]]:
        """Return one transformed ``(image, target)`` RF-DETR sample."""
        record = self._images.iloc[index]
        with Image.open(self._image_directory / record.file_name) as raw_image:
            image = raw_image.convert("RGB")
        width, height = image.size

        image_annotations = self._annotations_by_image.get(record.source_image_id)
        if image_annotations is None or image_annotations.empty:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            masks = torch.zeros((0, height, width), dtype=torch.uint8)
            areas = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.as_tensor(np.stack(image_annotations["bbox"].to_numpy()), dtype=torch.float32)
            boxes[:, 2:] += boxes[:, :2]
            boxes[:, 0::2] = boxes[:, 0::2].clamp(min=0, max=width)
            boxes[:, 1::2] = boxes[:, 1::2].clamp(min=0, max=height)
            masks = convert_coco_poly_to_mask(image_annotations["segmentation"].tolist(), height, width)
            areas = torch.as_tensor(image_annotations["area"].to_numpy(), dtype=torch.float32)

        keep = (boxes[:, 3] > boxes[:, 1]) & (boxes[:, 2] > boxes[:, 0])
        boxes, masks, areas = boxes[keep], masks[keep], areas[keep]
        num_instances = boxes.shape[0]
        target: dict[str, Any] = {
            "boxes": boxes,
            "labels": torch.zeros((num_instances,), dtype=torch.int64),
            "image_id": torch.as_tensor([index]),
            "area": areas,
            "iscrowd": torch.zeros((num_instances,), dtype=torch.int64),
            "masks": masks.bool(),
            "orig_size": torch.as_tensor([height, width]),
            "size": torch.as_tensor([height, width]),
        }
        if self._transforms is not None:
            image, target = self._transforms(image, target)
        return image, target


class FilamentDataModule(RFDETRDataModule):
    """Grouped RF-DETR DataModule built directly from the source annotation dataframes."""

    def __init__(
        self,
        model_config: RFDETRSegSmallConfig,
        train_config: SegmentationTrainConfig,
        image_directory: Path,
        images: pd.DataFrame,
        annotations: pd.DataFrame,
        validation_fraction: float,
        seed: int,
    ) -> None:
        super().__init__(model_config, train_config)
        if not 0 < validation_fraction < 1:
            raise ValueError(f"validation_fraction must be in (0, 1), got {validation_fraction}.")
        if images["file_name"].nunique() < 2:
            raise ValueError("Grouped validation requires at least two physical image names.")

        splitter = GroupShuffleSplit(n_splits=1, test_size=validation_fraction, random_state=seed)
        train_indices, validation_indices = next(splitter.split(images, groups=images["file_name"]))
        # Multi-annotator dedup (train only): ~64% of image records re-annotate the same physical JPEG with
        # deliberately disagreeing masks (MAGFiLO studies inter-annotator variability). Training on all copies
        # feeds the model identical pixels with conflicting targets, which it resolves by predicting blurry,
        # low-confidence masks. Keeping the single most-annotated record per JPEG gives one consistent target
        # per observation. Validation keeps every record so the proxy metric still averages over annotator
        # disagreement, exactly like the organizer's hidden test set may.
        self._train_images = (
            images.iloc[train_indices]
            .sort_values("annotation_count", ascending=False)
            .drop_duplicates("file_name", keep="first")
            .sample(frac=1.0, random_state=seed)
            .reset_index(drop=True)
        )
        self._validation_images = images.iloc[validation_indices].reset_index(drop=True)

        train_files = set(self._train_images["file_name"])
        validation_files = set(self._validation_images["file_name"])
        if train_files.intersection(validation_files):
            raise AssertionError("The grouped DataModule leaked a physical JPEG across train and validation.")

        self._annotations = annotations
        self._image_directory = image_directory

    @property
    def class_names(self) -> list[str]:
        """Return the only model class after collapsing annotation-side labels."""
        return ["filament"]

    @property
    def train_images(self) -> pd.DataFrame:
        """Expose the deterministic train image records for reporting."""
        return self._train_images

    @property
    def validation_images(self) -> pd.DataFrame:
        """Expose the deterministic validation image records for reporting and calibration."""
        return self._validation_images

    def _build_dataset(self, image_set: str, images: pd.DataFrame) -> FilamentDataset:
        """Build one RF-DETR dataset split with its split-specific transform."""
        transforms = make_coco_transforms_square_div_64(
            image_set,
            self.model_config.resolution,
            multi_scale=self.train_config.multi_scale,
            expanded_scales=self.train_config.expanded_scales,
            skip_random_resize=not self.train_config.do_random_resize_via_padding,
            patch_size=self.model_config.patch_size,
            num_windows=self.model_config.num_windows,
            aug_config=self.train_config.aug_config,
            gpu_postprocess=False,
        )
        return FilamentDataset(images, self._annotations, self._image_directory, transforms)

    def setup(self, stage: str | None = None) -> None:
        """Build lazy grouped datasets for Lightning fit, validation, test, or prediction stages."""
        if stage in (None, "fit"):
            if self._dataset_train is None:
                self._dataset_train = self._build_dataset("train", self._train_images)
            if self._dataset_val is None:
                self._dataset_val = self._build_dataset("val", self._validation_images)
        elif stage in {"validate", "predict"}:
            if self._dataset_val is None:
                self._dataset_val = self._build_dataset("val", self._validation_images)
        elif stage == "test" and self._dataset_test is None:
            self._dataset_test = self._build_dataset("test", self._validation_images)
        elif stage not in {"fit", "validate", "test", "predict"}:
            raise ValueError(f"Unsupported Lightning stage: {stage!r}")


module = RFDETRModelModule(model_config, train_config)
datamodule = FilamentDataModule(
    model_config,
    train_config,
    PATH_TRAIN_IMAGES,
    df_images,
    df_annotations,
    VALIDATION_FRACTION,
    SEED,
)
df_validation_images = datamodule.validation_images
df_split_summary = pd.concat(
    [datamodule.train_images.assign(split="train"), datamodule.validation_images.assign(split="valid")],
    ignore_index=True,
)
split_summary = (
    df_split_summary
    .groupby("split")
    .agg(
        image_records=("source_image_id", "size"),
        physical_images=("file_name", "nunique"),
        annotations=("annotation_count", "sum"),
    )
    .sort_index()
)
display(split_summary)
print("Finding — train/valid physical JPEG overlap: 0 (enforced in FilamentDataModule.__init__).")
datamodule.setup("fit")
samples, targets = next(iter(datamodule.train_dataloader()))

print(f"batch image tensor={tuple(samples.tensors.shape)} dtype={samples.tensors.dtype}")
print(f"first target masks={tuple(targets[0]['masks'].shape)} dtype={targets[0]['masks'].dtype}")
if (
    samples.tensors.ndim != 4
    or samples.tensors.shape[1] != model_config.num_channels
    or samples.tensors.dtype != torch.float32
):
    raise TypeError("RF-DETR image batches must be float32 tensors shaped (B, 3, H, W).")
if targets[0]["masks"].ndim != 3 or targets[0]["masks"].dtype != torch.bool:
    raise TypeError("RF-DETR instance masks must be three-dimensional boolean tensors.")

In [ ]:
BATCH_VISUALIZATION_N = 6
# Reuse RF-DETR's own private diagnostic figure (box + mask overlays on a real transformed batch, with a proper
# ImageNet un-normalize) instead of a hand-rolled plot.
figure = datamodule._show_samples(BATCH_VISUALIZATION_N, split="train")
figure.suptitle(f"Transformed training batch (masks + boxes), n={BATCH_VISUALIZATION_N}")
plt.show()

In [ ]:
# `accelerator="auto"`/`devices="auto"` lets Lightning select CUDA on Kaggle, Apple MPS on a local Mac, or CPU —
# so the same script runs unchanged locally and on Kaggle. Under SMOKE_RUN, cap each epoch to a handful of
# train/val batches (forwarded to the Lightning Trainer via build_trainer's **trainer_kwargs).
smoke_trainer_kwargs = (
    {"limit_train_batches": SMOKE_BATCHES, "limit_val_batches": SMOKE_BATCHES} if SMOKE_RUN else {}
)
trainer = build_trainer(
    train_config,
    model_config,
    # accelerator="auto",
    # devices="auto",
    fast_dev_run=FAST_DEV_RUN,
    **smoke_trainer_kwargs,
)
trainer.callbacks.append(LearningRateMonitor(logging_interval="step"))
trainer.fit(module, datamodule)

PATH_CHECKPOINT = PATH_MODELS / "checkpoint_best_total.pth"
if not FAST_DEV_RUN and not PATH_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Expected RF-DETR best checkpoint: {PATH_CHECKPOINT}")
print(f"best checkpoint={PATH_CHECKPOINT}")

In [ ]:
PATH_METRICS = sorted(PATH_MODELS.rglob("metrics.csv"))[-1]

### Why `plot_metrics`, not a hand-rolled plot

RF-DETR's `CSVLogger` writes sparse rows: per-step train metrics and per-epoch validation metrics never share a
row, and metrics logged with `on_step=on_epoch=True` arrive suffixed (`train/loss_step`, `train/loss_epoch`).
Plotting the raw frame naively against `epoch` therefore draws mostly-NaN columns and renders an empty chart.
`plot_metrics` already handles this — epoch-averaging, suffix-robust column discovery, dropping the synthetic
post-fit validation epoch — so it replaces a hand-rolled plot here instead of reimplementing the same fix.

In [ ]:
plot_metrics(str(PATH_METRICS), loss_log_scale=True)
plt.show()

## 7. Validation inference, mask refinement, and Dice-proxy threshold calibration

- Checks both the in-memory trained module and the portable best checkpoint.
- **Intensity-guided refinement**: snaps each coarse upsampled mask onto the dark filament pixels it covers
  (filaments are dark absorption features on a bright disk), recovering the thin-structure precision the
  mask-head grid cannot represent.
- **Single prediction pass**: each calibration image is predicted once at the lowest grid threshold; the sweep
  then just filters cached masks by confidence, so a 10-point grid costs the same inference as one point.
- Calibrates confidence using one-to-one matched mask Dice on a fixed validation sample, reported for raw and
  refined masks side by side — the gap is the measured value of refinement.
- **Proxy, not the real score**: organizers also score fragmentation, over-segmentation, and end-to-end
  efficiency — this is deliberately called a proxy.
- The calibrated threshold is carried unchanged into test prediction to avoid tuning on hidden data.

In [ ]:
CONFIDENCE_GRID = [round(float(t), 2) for t in np.arange(0.05, 0.55, 0.05)]
PREDICTION_FLOOR = min(CONFIDENCE_GRID)
CALIBRATION_IMAGE_N = 4 if SMOKE_RUN else 48
REFINE_MARGIN_PX = 12


def restore_binary_mask(mask: np.ndarray, height: int, width: int) -> np.ndarray:
    """Restore one binary instance mask to a required image size without soft interpolation."""
    if mask.ndim != 2:
        raise ValueError(f"Expected a two-dimensional instance mask, got shape {mask.shape}.")
    restored = Image.fromarray(mask.astype(np.uint8)).resize((width, height), resample=Image.Resampling.NEAREST)
    return np.asarray(restored, dtype=bool)


def predict_masks_and_scores(
    model: RFDETRSegSmall,
    rgb_image: Image.Image,
    threshold: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Return thresholded instance masks and their confidences for one already-RGB image."""
    detections = model.predict(rgb_image, threshold=threshold)
    masks = detections.mask
    if masks is None or len(masks) == 0:
        return np.empty((0, 0, 0), dtype=bool), np.empty((0,), dtype=np.float32)
    masks = np.asarray(masks, dtype=bool)
    scores = (
        np.asarray(detections.confidence, dtype=np.float32)
        if detections.confidence is not None
        else np.ones(len(masks), dtype=np.float32)
    )
    return masks, scores


def predict_instance_masks(model: RFDETRSegSmall, image_path: Path, threshold: float) -> np.ndarray:
    """Return thresholded RF-DETR instance masks for one image file.

    GONG H-Alpha JPEGs are single-channel grayscale; they are replicated to three channels so the ImageNet-pretrained
    RF-DETR backbone receives its expected RGB input without introducing false color information.
    """
    with Image.open(image_path) as image:
        rgb_image = image.convert("RGB")
    masks, _ = predict_masks_and_scores(model, rgb_image, threshold)
    return masks


def decode_target_masks(annotations: pd.DataFrame, height: int, width: int) -> np.ndarray:
    """Decode a dataframe of COCO polygons into one binary mask per filament."""
    masks: list[np.ndarray] = []
    for segmentation in annotations["segmentation"]:
        rles = coco_mask.frPyObjects(segmentation, height, width)
        decoded = coco_mask.decode(rles)
        if decoded.ndim == 3:
            decoded = decoded.any(axis=2)
        masks.append(decoded.astype(bool))
    if not masks:
        return np.empty((0, height, width), dtype=bool)
    return np.stack(masks)


def pairwise_dice_matrix(predicted_masks: list[np.ndarray], target_masks: np.ndarray) -> np.ndarray:
    """Compute the (predicted x target) pairwise Dice matrix for one image."""
    dice_matrix = np.zeros((len(predicted_masks), len(target_masks)), dtype=float)
    for predicted_index, predicted_mask in enumerate(predicted_masks):
        for target_index, target_mask in enumerate(target_masks):
            intersection = np.logical_and(predicted_mask, target_mask).sum()
            total_area = predicted_mask.sum() + target_mask.sum()
            dice_matrix[predicted_index, target_index] = 2.0 * intersection / max(total_area, 1)
    return dice_matrix


def matched_dice_from_matrix(dice_matrix: np.ndarray) -> float:
    """Measure mean Dice after overlap-maximizing one-to-one matching on a precomputed pairwise matrix.

    Dividing the matched-pair sum by ``max(predicted, target)`` charges every unmatched prediction and every
    missed filament a Dice of zero — a deliberate stand-in for the organizer's one-to-many/many-to-one penalties.
    """
    predicted_count, target_count = dice_matrix.shape
    if predicted_count == 0 and target_count == 0:
        return 1.0
    if predicted_count == 0 or target_count == 0:
        return 0.0
    predicted_indices, target_indices = linear_sum_assignment(-dice_matrix)
    return float(dice_matrix[predicted_indices, target_indices].sum() / max(predicted_count, target_count))

### Why intensity-guided mask refinement

The mask head predicts on a `resolution / mask_downsample_ratio` grid and the logits are bilinearly upsampled
to 2048px (section 5) — so even a perfect prediction arrives with blurred, blocky boundaries several pixels
wide, and on filaments that are themselves only a few pixels wide that blur alone erases most of the Dice
score. The physics offers a cheap fix: filaments are *dark absorption features on a bright disk*. RF-DETR is
kept for what a detector is good at — finding filaments and separating instances — and each coarse mask is then
snapped onto the dark pixels it covers:
#
1. Crop to the mask's bounding box (plus margin) so all work is local and fast.
2. Split the local intensities into their dark (filament) and bright (disk) modes with an Otsu threshold
   computed on the search region around the prediction — a per-filament darkness cut-off, robust to the
   disk's global brightness gradient.
3. Keep dark pixels within a small dilation of the predicted mask, but only connected components that touch
   the original prediction (stray dark noise nearby stays out).
4. Close single-pixel gaps; on any degenerate outcome fall back to the unrefined mask.
#
The calibration below reports raw vs refined Dice side by side, so the gain is measured, not assumed.

In [ ]:
def otsu_threshold(values: np.ndarray) -> float:
    """Find the intensity cut that best separates a dark and a bright mode (Otsu's method, 8-bit inputs)."""
    histogram = np.bincount(values.ravel(), minlength=256).astype(np.float64)
    bin_centers = np.arange(256, dtype=np.float64)
    weight_dark = np.cumsum(histogram)
    weight_bright = weight_dark[-1] - weight_dark
    cumulative_intensity = np.cumsum(histogram * bin_centers)
    mean_dark = cumulative_intensity / np.maximum(weight_dark, 1)
    mean_bright = (cumulative_intensity[-1] - cumulative_intensity) / np.maximum(weight_bright, 1)
    between_class_variance = weight_dark * weight_bright * (mean_dark - mean_bright) ** 2
    return float(np.argmax(between_class_variance))


def refine_mask_by_intensity(
    mask: np.ndarray,
    image_gray: np.ndarray,
    margin: int = REFINE_MARGIN_PX,
) -> np.ndarray:
    """Snap one coarse upsampled instance mask onto the dark H-Alpha filament pixels it covers."""
    rows = np.flatnonzero(mask.any(axis=1))
    columns = np.flatnonzero(mask.any(axis=0))
    if rows.size == 0:
        return mask
    row_lo, row_hi = max(int(rows[0]) - margin, 0), min(int(rows[-1]) + margin + 1, mask.shape[0])
    col_lo, col_hi = max(int(columns[0]) - margin, 0), min(int(columns[-1]) + margin + 1, mask.shape[1])
    crop_mask = mask[row_lo:row_hi, col_lo:col_hi]
    crop_gray = image_gray[row_lo:row_hi, col_lo:col_hi]

    search_region = ndimage.binary_dilation(crop_mask, iterations=max(margin // 2, 1))
    darkness_cutoff = otsu_threshold(crop_gray[search_region])
    dark_pixels = (crop_gray <= darkness_cutoff) & search_region
    component_labels, component_count = ndimage.label(dark_pixels)
    if component_count == 0:
        return mask
    touching = np.unique(component_labels[crop_mask & (component_labels > 0)])
    refined_crop = np.isin(component_labels, touching[touching > 0])
    refined_crop = ndimage.binary_closing(refined_crop, structure=np.ones((3, 3), dtype=bool))
    if not refined_crop.any():
        return mask

    refined = np.zeros_like(mask)
    refined[row_lo:row_hi, col_lo:col_hi] = refined_crop
    return refined


in_memory_model = RFDETRSegSmall(
    num_classes=1,
    pretrain_weights=PRETRAINED_WEIGHTS,
    device=str(DEVICE),
)
in_memory_model.model.model = module.model.to(DEVICE)
in_memory_model.model.class_names = ["filament"]
if DEVICE.type == "cuda":
    in_memory_model.optimize_for_inference(compile=False, dtype=torch.float16)
smoke_record = df_validation_images.iloc[0]
in_memory_masks = predict_instance_masks(
    in_memory_model,
    PATH_TRAIN_IMAGES / smoke_record.file_name,
    threshold=0.50,
)
print(f"in-memory prediction: masks={in_memory_masks.shape}; image={smoke_record.file_name}")

### Why the reloaded checkpoint can predict differently

`checkpoint_best_total.pth` is the best-validation-epoch checkpoint (`BestModelCallback`), not necessarily the
final/last epoch that `in_memory_model`'s live weights (`module.model`) represent — `early_stopping_patience` can
run training well past the best epoch, and with `use_ema=True` the saved winner may additionally be the EMA
weights rather than the live ones. A short/early training run can legitimately have zero masks above a fixed 0.50
threshold here even though the pipeline is healthy — the next section calibrates the real operating threshold on
this exact checkpoint, so a fixed spot-check at 0.50 is not the final answer.

In [ ]:
reloaded_model = RFDETRSegSmall(pretrain_weights=str(PATH_CHECKPOINT), device=str(DEVICE))
if DEVICE.type == "cuda":
    reloaded_model.optimize_for_inference(compile=False, dtype=torch.float16)
reloaded_masks = predict_instance_masks(
    reloaded_model,
    PATH_TRAIN_IMAGES / smoke_record.file_name,
    threshold=0.50,
)
print(f"reloaded prediction: masks={reloaded_masks.shape}; image={smoke_record.file_name}")
if reloaded_masks.shape[0] == 0:
    print("note — 0 masks at threshold=0.50 on this checkpoint; not an error, see comment above. Visualizing at a")
    print("lower diagnostic threshold below; the calibration section finds the real operating threshold.")

DIAGNOSTIC_THRESHOLD = 0.05
visualization_masks, visualization_threshold = (
    (reloaded_masks, 0.50)
    if reloaded_masks.shape[0] > 0
    else (
        predict_instance_masks(reloaded_model, PATH_TRAIN_IMAGES / smoke_record.file_name, DIAGNOSTIC_THRESHOLD),
        DIAGNOSTIC_THRESHOLD,
    )
)
figure, axis = plt.subplots(figsize=(7, 7))
with Image.open(PATH_TRAIN_IMAGES / smoke_record.file_name) as image:
    axis.imshow(image, cmap="gray")
for mask in visualization_masks:
    axis.imshow(restore_binary_mask(mask, IMAGE_HEIGHT, IMAGE_WIDTH), cmap="autumn", alpha=0.25)
axis.set_title(f"Reloaded RF-DETR-Seg prediction (threshold={visualization_threshold:.2f})")
axis.axis("off")
plt.tight_layout()
plt.show()

### Learned crop-U-Net refiner (replaces the intensity heuristic as the active refiner)

The Otsu intensity refiner above is a fixed physics heuristic: it snaps a coarse mask onto the dark pixels it
covers. It helps on clean, well-placed masks but amplifies localisation error where the coarse mask is off or the
local histogram is not bimodal (limb darkening, sunspots) — a variance amplifier, not a robust corrector. A
reference Mask R-CNN + U-Net-refiner solution scored ~0.69 *with* a refiner, so refinement is not the problem;
the *hand-crafted* refiner is. This section ports that learned refiner: a ResNet18-encoder U-Net trained on
square image crops around each ground-truth filament to predict the fine mask. At inference each detector mask is
cropped (square, context-padded), passed through the U-Net, thresholded, and pasted back at full resolution.
`refine_masks_with_unet` is the single refinement entry point used by calibration, the semantic lens, and the
submission, so the validation proxy now measures exactly the masks that get submitted.

In [ ]:
CROP_SIZE = 256
REFINER_CONTEXT = 1.8  # square crop side = filament bbox side * context (competitor default)
REFINER_MIN_SIDE = 96  # smallest crop side in pixels, so tiny filaments still fill the input
REFINER_THRESHOLD = 0.85  # sigmoid cut on the refined probability map
REFINER_EPOCHS = 1 if SMOKE_RUN else 8
REFINER_BATCH = 16
REFINER_MAX_CROPS = 128 if SMOKE_RUN else None  # cap crop cache under smoke to keep the path fast
PATH_REFINER = PATH_MODELS / "crop_refiner.pt"


def square_bounds(
    mask: np.ndarray, context: float = REFINER_CONTEXT, minimum: int = REFINER_MIN_SIDE
) -> tuple[int, int, int, int]:
    """Return a square, context-padded, in-bounds crop box (x0, y0, x1, y1) around a binary mask."""
    height, width = mask.shape
    rows, columns = np.where(mask)
    center_x, center_y = (columns.min() + columns.max() + 1) / 2, (rows.min() + rows.max() + 1) / 2
    side = int(np.ceil(max(columns.max() - columns.min() + 1, rows.max() - rows.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = round(center_x - side / 2), round(center_y - side / 2)
    x1, y1 = x0 + side, y0 + side
    if x0 < 0:
        x1 -= x0
        x0 = 0
    if y0 < 0:
        y1 -= y0
        y0 = 0
    if x1 > width:
        x0 -= x1 - width
        x1 = width
    if y1 > height:
        y0 -= y1 - height
        y1 = height
    return int(x0), int(y0), int(x1), int(y1)


class ConvBlock(nn.Sequential):
    """Two 3x3 conv-BN-ReLU layers, the standard U-Net decoder building block."""

    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


class CropUNet(nn.Module):
    """ResNet18-encoder U-Net that predicts a fine filament mask from one square image crop."""

    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = (
            encoder.maxpool, encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4
        )
        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)
        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        """Return per-pixel mask logits for a (B, 3, CROP_SIZE, CROP_SIZE) batch of normalised crops."""
        image = (image - self.mean) / self.std
        x0 = self.stem(image)
        x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        x = self.dec4(torch.cat([self.up4(x4), x3], 1))
        x = self.dec3(torch.cat([self.up3(x), x2], 1))
        x = self.dec2(torch.cat([self.up2(x), x1], 1))
        x = self.dec1(torch.cat([self.up1(x), x0], 1))
        return self.head(self.up0(x))


def soft_dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Return the smoothed soft-Dice loss between mask logits and binary targets."""
    probabilities = logits.sigmoid()
    intersection = (probabilities * targets).sum((1, 2, 3))
    denominator = probabilities.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()


class CropRefinerDataset(Dataset):
    """Serve (3-channel crop, single-channel target) pairs with optional geometric/photometric augmentation."""

    def __init__(self, images: np.ndarray, masks: np.ndarray, augment: bool = False) -> None:
        self.images, self.masks, self.augment = images, masks, augment

    def __len__(self) -> int:
        """Return the number of cached crops."""
        return len(self.images)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Return the augmented (3-channel crop, single-channel target) pair at ``index``."""
        image = torch.from_numpy(self.images[index].astype(np.float32) / 255.0)
        mask = torch.from_numpy(self.masks[index].astype(np.float32)).unsqueeze(0)
        if self.augment:
            rotations = int(torch.randint(0, 4, (1,)).item())
            image, mask = torch.rot90(image, rotations, (0, 1)), torch.rot90(mask, rotations, (1, 2))
            if torch.rand(1).item() < 0.5:
                image, mask = torch.flip(image, (1,)), torch.flip(mask, (2,))
            if torch.rand(1).item() < 0.5:
                image = image ** float(torch.empty(1).uniform_(0.85, 1.20).item())
            if torch.rand(1).item() < 0.5:
                image = torch.clamp(image * float(torch.empty(1).uniform_(0.90, 1.10).item()), 0, 1)
        return image.unsqueeze(0).repeat(3, 1, 1), mask


def build_refiner_crops(
    image_records: pd.DataFrame, annotations: pd.DataFrame, max_crops: int | None
) -> tuple[np.ndarray, np.ndarray]:
    """Cache one square (image, ground-truth-mask) crop per annotated filament, resized to CROP_SIZE."""
    crop_images: list[np.ndarray] = []
    crop_masks: list[np.ndarray] = []
    for record in tqdm(image_records.itertuples(index=False), total=len(image_records), desc="refiner crops"):
        height, width = int(record.height), int(record.width)
        target_masks = decode_target_masks(
            annotations.loc[annotations["source_image_id"] == record.source_image_id], height=height, width=width
        )
        if len(target_masks) == 0:
            continue
        with Image.open(PATH_TRAIN_IMAGES / record.file_name) as image:
            image_gray = np.asarray(image.convert("L"))
        for target_mask in target_masks:
            x0, y0, x1, y1 = square_bounds(target_mask)
            mask_crop = target_mask[y0:y1, x0:x1].astype(np.uint8)
            crop_images.append(
                cv2.resize(image_gray[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR)
            )
            crop_masks.append(
                cv2.resize(mask_crop, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_NEAREST)
            )
            if max_crops is not None and len(crop_images) >= max_crops:
                return np.stack(crop_images), np.stack(crop_masks)
    if not crop_images:
        raise RuntimeError("No ground-truth filament crops were built for the refiner.")
    return np.stack(crop_images), np.stack(crop_masks)


def evaluate_refiner_dice(model: CropUNet, loader: DataLoader, device: torch.device) -> float:
    """Return mean per-crop Dice on a validation loader at the 0.65 operating point."""
    model.eval()
    dice_total = samples = 0.0
    with torch.inference_mode():
        for images, masks in loader:
            masks = masks.to(device, non_blocking=True).bool()
            predictions = model(images.to(device, non_blocking=True)).sigmoid() >= 0.65
            intersection = (predictions & masks).sum((1, 2, 3)).float()
            denominator = predictions.sum((1, 2, 3)) + masks.sum((1, 2, 3))
            dice_total += ((2 * intersection + 1e-7) / (denominator + 1e-7)).sum().item()
            samples += len(images)
    return dice_total / max(samples, 1)


def train_crop_refiner(
    model: CropUNet,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    output_path: Path,
    device: torch.device,
) -> float:
    """Train the crop refiner with a 0.5 BCE + 0.5 soft-Dice loss; checkpoint the best validation Dice."""
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best_dice = -1.0
    for epoch in range(1, epochs + 1):
        model.train()
        for images, masks in tqdm(train_loader, desc=f"refiner epoch {epoch:02d}/{epochs}", leave=False):
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(images)
                bce = nn.functional.binary_cross_entropy_with_logits(logits, masks)
                loss = 0.5 * bce + 0.5 * soft_dice_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        val_dice = evaluate_refiner_dice(model, val_loader, device)
        if val_dice >= best_dice:
            best_dice = val_dice
            torch.save({"epoch": epoch, "model": model.state_dict()}, output_path)
        print(f"refiner epoch {epoch:02d}/{epochs} | val Dice {val_dice:.4f}")
    return best_dice

In [ ]:
refiner_train_crops = build_refiner_crops(datamodule.train_images, df_annotations, REFINER_MAX_CROPS)
refiner_val_crops = build_refiner_crops(datamodule.validation_images, df_annotations, REFINER_MAX_CROPS)
refiner_train_loader = DataLoader(
    CropRefinerDataset(*refiner_train_crops, augment=True),
    batch_size=REFINER_BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
)
refiner_val_loader = DataLoader(
    CropRefinerDataset(*refiner_val_crops, augment=False),
    batch_size=REFINER_BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
)
print(f"refiner crops — train={len(refiner_train_loader.dataset):,}; val={len(refiner_val_loader.dataset):,}")

crop_refiner = CropUNet(pretrained=True)
best_refiner_dice = train_crop_refiner(
    crop_refiner, refiner_train_loader, refiner_val_loader, REFINER_EPOCHS, PATH_REFINER, DEVICE
)
crop_refiner.load_state_dict(torch.load(PATH_REFINER, map_location=DEVICE, weights_only=True)["model"])
crop_refiner.to(DEVICE).eval()
print(f"crop refiner trained; best val crop-Dice={best_refiner_dice:.4f}; weights={PATH_REFINER}")

In [ ]:
def refine_masks_with_unet(
    masks: list[np.ndarray],
    image_gray: np.ndarray,
    refiner: CropUNet | None,
    device: torch.device = DEVICE,
    threshold: float = REFINER_THRESHOLD,
) -> list[np.ndarray]:
    """Refine each full-resolution instance mask with the crop U-Net, falling back to the input mask on failure."""
    refined = [mask.copy() for mask in masks]
    if refiner is None or not masks:
        return refined
    height, width = image_gray.shape
    crops: list[np.ndarray] = []
    bounds: list[tuple[int, int, int, int]] = []
    indices: list[int] = []
    for index, mask in enumerate(masks):
        if not mask.any():
            continue
        x0, y0, x1, y1 = square_bounds(mask)
        crops.append(cv2.resize(image_gray[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR))
        bounds.append((x0, y0, x1, y1))
        indices.append(index)
    if not crops:
        return refined
    crop_tensor = torch.from_numpy(np.stack(crops)).float().unsqueeze(1).repeat(1, 3, 1, 1).to(device) / 255.0
    with torch.inference_mode():
        probabilities = refiner(crop_tensor).sigmoid().cpu().numpy()[:, 0]
    for index, probability, (x0, y0, x1, y1) in zip(indices, probabilities, bounds, strict=True):
        crop_mask = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) >= threshold
        if crop_mask.any():
            full = np.zeros((height, width), dtype=bool)
            full[y0:y1, x0:x1] = crop_mask
            refined[index] = full
    return refined

In [ ]:
calibration_records = df_validation_images.sample(
    n=min(CALIBRATION_IMAGE_N, len(df_validation_images)),
    random_state=SEED,
)

GALLERY_IMAGE_N = 4

# One prediction pass per image at the grid floor; the threshold sweep below filters these per-image summaries.
# Only tiny arrays are kept for every image — the confidence scores and the raw/refined pairwise-Dice matrices
# (a few floats each). The full-resolution 2048x2048 instance masks are deliberately NOT retained per image:
# holding raw+refined masks for all calibration images pinned several GB of masks in RAM and OOM-killed the later
# full-resolution test-inference loop (no Python traceback — a cgroup kill, which Kaggle reports only as
# "Failed"). Full masks are kept for just the first GALLERY_IMAGE_N images, for the visual check below.
#
# NOTE (possible future optimization — `supervision.CompactMask`): the semantic-Dice lens below re-runs inference
# because full masks are not cached here. To instead cache every image's masks across the whole section without
# the OOM that caused, store them as `supervision.CompactMask` (crop-RLE, O(N * bbox_area) rather than
# O(N * H * W) — a large win for these thin, sparse filaments) via `CompactMask.from_dense(...)`, then
# `.to_dense()` per image only where an ndarray op is needed. Not adopted now to keep the pipeline dense and
# simple; `RFDETRSegSmall.predict` already returns dense masks, so the win only materializes if masks are held
# resident across many images (exactly what this loop avoids).
calibration_cache: list[dict[str, Any]] = []
gallery_cache: list[dict[str, Any]] = []
for image_row in tqdm(calibration_records.itertuples(index=False), desc="calibration inference"):
    height, width = int(image_row.height), int(image_row.width)
    target_masks = decode_target_masks(
        df_annotations.loc[df_annotations["source_image_id"] == image_row.source_image_id],
        height=height,
        width=width,
    )
    with Image.open(PATH_TRAIN_IMAGES / image_row.file_name) as image:
        image_gray = np.asarray(image.convert("L"))
        rgb_image = image.convert("RGB")
    floor_masks, floor_scores = predict_masks_and_scores(reloaded_model, rgb_image, PREDICTION_FLOOR)
    raw_masks = [restore_binary_mask(mask, height, width) for mask in floor_masks]
    refined_masks = refine_masks_with_unet(raw_masks, image_gray, crop_refiner)
    calibration_cache.append({
        "scores": floor_scores,
        "dice_raw": pairwise_dice_matrix(raw_masks, target_masks),
        "dice_refined": pairwise_dice_matrix(refined_masks, target_masks),
    })
    if len(gallery_cache) < GALLERY_IMAGE_N:
        gallery_cache.append({
            "file_name": image_row.file_name,
            "scores": floor_scores,
            "raw_masks": raw_masks,
            "refined_masks": refined_masks,
            "target_masks": target_masks,
        })
    # raw_masks / refined_masks / target_masks leave scope here and are freed unless captured for the gallery.

calibration_rows = [
    {
        "threshold": threshold,
        "matched_dice_raw": float(
            np.mean([matched_dice_from_matrix(c["dice_raw"][c["scores"] >= threshold]) for c in calibration_cache])
        ),
        "matched_dice_refined": float(
            np.mean([matched_dice_from_matrix(c["dice_refined"][c["scores"] >= threshold]) for c in calibration_cache])
        ),
    }
    for threshold in CONFIDENCE_GRID
]

df_calibration = pd.DataFrame(calibration_rows).sort_values("threshold")
display(df_calibration)
CALIBRATED_THRESHOLD = float(df_calibration.loc[df_calibration["matched_dice_refined"].idxmax(), "threshold"])
axis = df_calibration.plot(x="threshold", y=["matched_dice_raw", "matched_dice_refined"], marker="o", figsize=(8, 4))
axis.axvline(CALIBRATED_THRESHOLD, color="tab:red", linestyle="--", alpha=0.6)
axis.set_title("Grouped-validation matched-instance Dice proxy (raw vs intensity-refined masks)")
axis.set_xlabel("RF-DETR confidence threshold")
axis.set_ylabel("Mean matched-instance Dice")
axis.grid(True)
plt.show()
best_row = df_calibration.loc[df_calibration["matched_dice_refined"].idxmax()]
print(
    f"Selected threshold={CALIBRATED_THRESHOLD:.2f}; refinement gain at that threshold: "
    f"{best_row['matched_dice_raw']:.4f} -> {best_row['matched_dice_refined']:.4f} matched Dice."
)
print("This is a validation proxy, not the organizer's full score.")

### Visual check — ground truth vs raw vs refined predictions

Numbers hide failure modes; morphology is 30% of the organizer's rubric. Each row below shows one calibration
image: expert ground truth (green), raw RF-DETR masks at the calibrated threshold (red), and the same masks
after intensity refinement (blue). What to look for: refined masks should trace the dark filament threads and
recover barbs; fragmented or merged instances show up here long before they show up in the score.

In [ ]:
def overlay_instance_masks(axis: plt.Axes, image_gray: np.ndarray, masks: list[np.ndarray], color: str) -> None:
    """Draw one grayscale image with a semi-transparent union overlay of the given instance masks."""
    axis.imshow(image_gray, cmap="gray")
    if masks:
        union = np.logical_or.reduce(masks)
        overlay = np.zeros((*union.shape, 4), dtype=np.float32)
        overlay[union] = plt.matplotlib.colors.to_rgba(color, alpha=0.45)
        axis.imshow(overlay)
    axis.axis("off")


figure, axes = plt.subplots(len(gallery_cache), 3, figsize=(15, 5 * len(gallery_cache)))
for row_index, entry in enumerate(gallery_cache):
    keep = entry["scores"] >= CALIBRATED_THRESHOLD
    kept_raw = [mask for mask, kept in zip(entry["raw_masks"], keep, strict=True) if kept]
    kept_refined = [mask for mask, kept in zip(entry["refined_masks"], keep, strict=True) if kept]
    with Image.open(PATH_TRAIN_IMAGES / entry["file_name"]) as image:
        image_gray = np.asarray(image.convert("L"))
    panels = [
        (list(entry["target_masks"]), "tab:green", f"ground truth (n={len(entry['target_masks'])})"),
        (kept_raw, "tab:red", f"raw prediction (n={len(kept_raw)})"),
        (kept_refined, "tab:blue", f"refined prediction (n={len(kept_refined)})"),
    ]
    for column_index, (panel_masks, color, title) in enumerate(panels):
        axis = np.atleast_2d(axes)[row_index, column_index]
        overlay_instance_masks(axis, image_gray, panel_masks, color)
        axis.set_title(f"{entry['file_name']} — {title}", fontsize=8)
plt.tight_layout()
plt.show()

### Semantic Dice cross-check with the organizer's named metric

- **Primary lens**: the matched-instance proxy above is the more faithful stand-in because the organizer
  penalizes fragmentation and one-to-many correspondences.
- **Complementary lens**: this reports the union-mask `torchmetrics.segmentation.DiceScore` named in the
  competition rubric.
- **What a gap means**: a large gap between the two Dice figures flags an instance-matching or
  over-segmentation problem rather than a pixel-overlap one.

In [ ]:
def foreground_onehot(union_mask: np.ndarray) -> torch.Tensor:
    """Encode one binary foreground mask as a one-hot ``(1, 2, H, W)`` boolean tensor."""
    foreground = torch.from_numpy(np.ascontiguousarray(union_mask, dtype=bool))
    return torch.stack([~foreground, foreground]).unsqueeze(0)


# Re-predict one image at a time at the calibrated threshold (not the floor), so only a single image's masks are
# ever in RAM — the calibration loop above deliberately did not retain them for every image (see its comment).
semantic_dice_metric = DiceScore(num_classes=2, include_background=False, input_format="one-hot")
for image_row in tqdm(calibration_records.itertuples(index=False), desc="semantic Dice lens"):
    height, width = int(image_row.height), int(image_row.width)
    target_masks = decode_target_masks(
        df_annotations.loc[df_annotations["source_image_id"] == image_row.source_image_id],
        height=height,
        width=width,
    )
    target_union = target_masks.any(axis=0) if len(target_masks) else np.zeros((height, width), dtype=bool)
    with Image.open(PATH_TRAIN_IMAGES / image_row.file_name) as image:
        image_gray = np.asarray(image.convert("L"))
        rgb_image = image.convert("RGB")
    predicted_masks, _ = predict_masks_and_scores(reloaded_model, rgb_image, CALIBRATED_THRESHOLD)
    restored_predicted = [restore_binary_mask(mask, height, width) for mask in predicted_masks]
    refined_masks = refine_masks_with_unet(restored_predicted, image_gray, crop_refiner)
    predicted_union = (
        np.logical_or.reduce(refined_masks) if refined_masks else np.zeros((height, width), dtype=bool)
    )
    semantic_dice_metric.update(foreground_onehot(predicted_union), foreground_onehot(target_union))

semantic_dice = float(semantic_dice_metric.compute())
print(f"Semantic union DiceScore @ threshold={CALIBRATED_THRESHOLD:.2f}: {semantic_dice:.4f} (torchmetrics, refined).")
print("Design implication — a semantic Dice far above the matched-instance proxy points to over-segmentation.")

## 8. RLE submission generation and validation

- **Row shape**: the competition accepts one CSV row per predicted filament, not a fixed row per test image.
- **Encoding**: restores each thresholded mask to the documented 2048x2048 grid, applies the same
  intensity-guided refinement that was calibrated in section 7, emits compressed COCO RLE counts only, and
  assigns a unique `image_id_1`, `image_id_2`, and so on.
- **TTA**: horizontal-flip test-time augmentation recovers filaments the model only detects in one orientation —
  label-safe here because the H-Alpha disk carries no left/right semantics. The two passes are merged with
  confidence-ranked IoU suppression so a filament seen in both passes isn't double-counted into the fragmentation
  penalty.
- **Validation**: reloads the written CSV and checks schema, uniqueness, source-image coverage, and RLE presence
  before reporting the output path.

In [ ]:
PATH_SUBMISSION = PATH_OUTPUT / "submission.csv"
USE_TTA = True
TTA_DEDUP_IOU = 0.5
# Refinement now runs the learned crop U-Net (section 7), not the Otsu heuristic that coincided with the
# 0.45 -> 0.12 drop. On applies the trained refiner to every submitted mask; set False to A/B against raw restored
# masks on a fixed detector checkpoint (the raw path that scored 0.45).
USE_REFINEMENT = True

### Why RLE encoding and IoU dedup

Submission masks must be compressed COCO RLE counts (organizer format, not polygons or PNGs). IoU dedup runs on
every image for two reasons: TTA runs each image through the model twice (below), and intensity refinement can
snap two distinct coarse masks onto the same dark filament thread — either way, without suppression one physical
filament would be emitted as two rows and penalized by the organizer's over-segmentation rubric.

In [ ]:
def encode_rle_counts(mask: np.ndarray) -> str:
    """Encode one 2048x2048 binary mask as the required compressed COCO RLE counts."""
    if mask.shape != (IMAGE_HEIGHT, IMAGE_WIDTH):
        raise ValueError(f"Submission mask has shape {mask.shape}, expected {(IMAGE_HEIGHT, IMAGE_WIDTH)}.")
    encoded = coco_mask.encode(np.asfortranarray(mask.astype(np.uint8)))
    return encoded["counts"].decode("ascii")


def mask_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Return the intersection-over-union of two aligned binary masks (0.0 when their union is empty)."""
    intersection = int(np.logical_and(mask_a, mask_b).sum())
    union = int(np.logical_or(mask_a, mask_b).sum())
    return intersection / union if union else 0.0


def deduplicate_masks(masks: list[np.ndarray], scores: np.ndarray, iou_threshold: float) -> list[np.ndarray]:
    """Greedy confidence-ranked suppression: keep a mask only if it barely overlaps every stronger kept mask."""
    kept: list[np.ndarray] = []
    for index in np.argsort(-scores):
        candidate = masks[int(index)]
        if all(mask_iou(candidate, keeper) < iou_threshold for keeper in kept):
            kept.append(candidate)
    return kept

### Why horizontal-flip TTA is label-safe here

Horizontal-flip TTA recovers filaments the model only detects in one orientation. This is label-safe here —
unlike most object classes, the H-Alpha solar disk has no inherent left/right semantics, so a flipped filament is
still a valid filament. Both passes are restored to the shared 2048x2048 output frame (flip mirrored back) before
dedup.

In [ ]:
def predict_submission_masks(model: RFDETRSegSmall, image_path: Path, threshold: float) -> list[np.ndarray]:
    """Predict full-resolution filament masks, optionally fusing a horizontal-flip TTA pass.

    When ``USE_REFINEMENT`` is set, each coarse mask is snapped onto the dark filament pixels it covers before
    deduplication (refine first, deduplicate second — two coarse masks that snap onto the same dark thread only
    become detectable duplicates after refinement). When it is off, the raw restored masks are submitted directly.
    """
    with Image.open(image_path) as image:
        image_gray = np.asarray(image.convert("L"))
        rgb_image = image.convert("RGB")
        flipped_image = rgb_image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    base_masks, base_scores = predict_masks_and_scores(model, rgb_image, threshold)
    restored = [restore_binary_mask(mask, IMAGE_HEIGHT, IMAGE_WIDTH) for mask in base_masks]
    scores = base_scores
    if USE_TTA:
        flip_masks, flip_scores = predict_masks_and_scores(model, flipped_image, threshold)
        # Restore, then mirror columns back to the original orientation so both passes share one coordinate frame.
        restored += [restore_binary_mask(mask, IMAGE_HEIGHT, IMAGE_WIDTH)[:, ::-1] for mask in flip_masks]
        scores = np.concatenate([base_scores, flip_scores])
    if not restored:
        return []
    candidate = refine_masks_with_unet(restored, image_gray, crop_refiner) if USE_REFINEMENT else restored
    return deduplicate_masks(candidate, scores, TTA_DEDUP_IOU)

### Detection decomposition — is the detector or the refiner the ceiling?

The matched-instance proxy is one number; it cannot say *why* the score stalls. This cell runs the exact
submission path (TTA + learned refiner + dedup, at the calibrated threshold) on the grouped-validation images and
splits the result into the two factors that multiply into the final Dice:
#
- **Instance recall** = matched predictions / ground-truth filaments. Low recall means the detector is missing
  filaments — a resolution / capacity / query-budget problem that no refiner can fix. This is the axis that would
  confirm an RF-DETR detector ceiling.
- **Matched-instance Dice** = mask quality on the filaments that *were* found. Low here (with high recall) means
  the refiner, not the detector, is the cap.
- **Precision** and **over-segmentation** (predictions per ground-truth) expose duplicate/false-positive masks
  that the organizer penalises — a dedup / threshold problem.
#
Decision rule: recall is the low number -> push the detector (resolution 768->1024, Small->Medium, or swap to a
Mask R-CNN); matched Dice is the low number -> push the refiner; over-seg high -> tighten dedup/threshold.

In [ ]:
def decompose_detection(predicted_masks: list[np.ndarray], target_masks: np.ndarray) -> dict[str, float]:
    """Split one image's masks into matched count, recall, precision, over-segmentation, and matched Dice."""
    num_pred, num_target = len(predicted_masks), len(target_masks)
    base = {"num_pred": float(num_pred), "num_target": float(num_target)}
    if num_pred == 0 or num_target == 0:
        return {
            **base, "matched": 0.0, "recall": 0.0, "precision": 0.0,
            "over_seg": num_pred / max(num_target, 1), "matched_dice": float("nan"),
        }
    dice_matrix = pairwise_dice_matrix(predicted_masks, target_masks)
    pred_indices, target_indices = linear_sum_assignment(-dice_matrix)
    pair_dice = dice_matrix[pred_indices, target_indices]
    hit = pair_dice > 1e-6
    matched = int(hit.sum())
    return {
        **base, "matched": float(matched),
        "recall": matched / num_target,
        "precision": matched / num_pred,
        "over_seg": num_pred / num_target,
        "matched_dice": float(pair_dice[hit].mean()) if matched else float("nan"),
    }


decomposition_rows = []
for image_row in tqdm(calibration_records.itertuples(index=False), desc="detection decomposition"):
    height, width = int(image_row.height), int(image_row.width)
    target_masks = decode_target_masks(
        df_annotations.loc[df_annotations["source_image_id"] == image_row.source_image_id], height=height, width=width
    )
    submitted_masks = predict_submission_masks(
        reloaded_model, PATH_TRAIN_IMAGES / image_row.file_name, CALIBRATED_THRESHOLD
    )
    decomposition_rows.append(decompose_detection(submitted_masks, target_masks))

df_decomposition = pd.DataFrame(decomposition_rows)
decomposition_summary = pd.Series({
    "instance recall": df_decomposition["recall"].mean(),
    "instance precision": df_decomposition["precision"].mean(),
    "over-seg (pred/GT)": df_decomposition["over_seg"].mean(),
    "matched-instance Dice": np.nanmean(df_decomposition["matched_dice"]),
})
display(decomposition_summary.round(4).to_frame("value"))
recall_value = float(decomposition_summary["instance recall"])
matched_dice_value = float(decomposition_summary["matched-instance Dice"])
over_seg_value = float(decomposition_summary["over-seg (pred/GT)"])
if recall_value < 0.70:
    verdict = "DETECTOR-bound — filaments are missed; push resolution/size or swap the detector (RF-DETR is the cap)."
elif matched_dice_value < 0.70:
    verdict = "REFINER-bound — filaments are found but masks are loose; push the refiner, not the detector."
elif over_seg_value > 1.30:
    verdict = "OVER-SEGMENTING — duplicate/false-positive masks; tighten dedup IoU or raise the score threshold."
else:
    verdict = "Balanced — no single dominant bottleneck; combined small gains needed."
print(f"decomposition verdict: {verdict}")

### Why row count varies per image

One inference pass per test image, flattened to one CSV row per predicted filament — the competition scores
per-filament, not a fixed row per image, so an image with 3 filaments contributes 3 rows and an empty image
contributes 0.

In [ ]:
test_paths = sorted({*PATH_TEST_IMAGES.glob("*.jpeg"), *PATH_TEST_IMAGES.glob("*.jpg")})
if not test_paths:
    raise FileNotFoundError(f"No JPEG test images found in {PATH_TEST_IMAGES}")
if SMOKE_RUN:
    # Smoke run only proves the inference+encode+validate path executes; a partial submission is expected.
    test_paths = test_paths[:8]

TEST_GALLERY_N = 3

submission_rows: list[dict[str, str]] = []
test_gallery: list[tuple[Path, list[np.ndarray]]] = []
for test_path in tqdm(test_paths, desc=f"test inference (tta={USE_TTA})"):
    prediction_masks = predict_submission_masks(reloaded_model, test_path, threshold=CALIBRATED_THRESHOLD)
    if len(test_gallery) < TEST_GALLERY_N:
        test_gallery.append((test_path, prediction_masks))
    for filament_number, prediction_mask in enumerate(prediction_masks, start=1):
        submission_rows.append({
            "filament_id": f"{test_path.stem}_{filament_number}",
            "segmentation_rle": encode_rle_counts(prediction_mask),
        })

### Visual check — submitted test predictions

The same visual lens as section 7, but on unseen test images and exactly the masks written into the CSV
(refined, TTA-fused, deduplicated). No ground truth exists here — judge morphology: masks should trace dark,
elongated filament threads, not blobs or scattered noise.

In [ ]:
figure, axes = plt.subplots(1, len(test_gallery), figsize=(6 * len(test_gallery), 6))
for axis, (gallery_path, gallery_masks) in zip(np.atleast_1d(axes).ravel(), test_gallery, strict=True):
    with Image.open(gallery_path) as image:
        gallery_gray = np.asarray(image.convert("L"))
    overlay_instance_masks(axis, gallery_gray, gallery_masks, "tab:blue")
    axis.set_title(f"{gallery_path.name} — {len(gallery_masks)} filaments submitted", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Round-trip through disk once, then validate the reloaded file (not just the in-memory dataframe) — this catches
# CSV-serialization issues (e.g. a stray comma corrupting RLE counts) that an in-memory check would miss.
submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
submission.to_csv(PATH_SUBMISSION, index=False)
reloaded_submission = pd.read_csv(PATH_SUBMISSION)

expected_columns = ["filament_id", "segmentation_rle"]
if reloaded_submission.columns.tolist() != expected_columns:
    raise AssertionError(f"Submission columns differ: {reloaded_submission.columns.tolist()}")
if not reloaded_submission["filament_id"].is_unique:
    raise AssertionError("Submission filament ids must be unique.")
if not reloaded_submission.empty and not reloaded_submission["segmentation_rle"].str.len().gt(0).all():
    raise AssertionError("Every predicted filament requires non-empty RLE counts.")

submitted_image_ids = set(reloaded_submission["filament_id"].str.rsplit("_", n=1).str[0])
test_image_ids = {test_path.stem for test_path in test_paths}
if not submitted_image_ids.issubset(test_image_ids):
    raise AssertionError("A submission row references an unknown test image id.")

In [ ]:
display(reloaded_submission.head())
print(f"test images={len(test_paths):,}; predicted filaments={len(reloaded_submission):,}")
print(f"submission={PATH_SUBMISSION.resolve()}")
# ! head "{PATH_SUBMISSION}"